# G2 matched rerun and gate-behaviour analysis

This notebook trains only `G2_matched_rerun` on the same ten locked patient folds and with the same saved preprocessing, seed schedule, loss and stopping rules used by G0, G1 and G3. The original G2 and completed controls are read only. The untouched final-test cohort is never evaluated.

Keep this notebook, `QGECAF_G2_Matched_Rerun_and_Gate_Analysis.py`, and `QGECAF_Reviewer_Revision_Nested_BGWO.py` in the same folder. Restart the kernel before running it.

In [2]:
qgecaf_root = (
    r"D:\chaitali\qgacef"
    r"\QGECAF_REVIEWER_REVISION_NESTED_BGWO"
)

controls_root = (
    r"D:\chaitali\qgacef"
    r"\QGECAF_MATCHED_GATE_CONTROLS"
)

output = (
    r"D:\chaitali\qgacef"
    r"\QGECAF_G2_MATCHED_RERUN"
)

In [3]:
import os
os.environ['TF_DETERMINISTIC_OPS'] = '1'

from pathlib import Path
import pandas as pd
import tensorflow as tf
from QGECAF_G2_Matched_Rerun_and_Gate_Analysis import Config, run

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

TensorFlow: 2.10.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
package_root = Path.cwd()
cfg = Config(
    qgecaf_root=(
        r'D:\chaitali\qgacef'
        r'\QGECAF_REVIEWER_REVISION_NESTED_BGWO'
    ),
    qgecaf_source=str(package_root / 'QGECAF_Reviewer_Revision_Nested_BGWO.py'),
    controls_root=(
        r'D:\chaitali\qgacef'
        r'\QGECAF_MATCHED_GATE_CONTROLS'
    ),
    output=(
        r'D:\chaitali\qgacef'
        r'\QGECAF_G2_MATCHED_RERUN'
    ),
    image_size=128,
    num_classes=6,
    outer_folds=10,
    seed=42,
    embed_dim=64,
    attention_heads=4,
    batch_size=8,
    epochs=50,
    learning_rate=1e-4,
    kl_weight=1e-3,
    dropout=0.30,
    fine_tune_last=30,
    early_stopping_patience=8,
    gate_hidden_units=16,
    resume=True,
)
cfg

Config(qgecaf_root='D:\\chaitali\\qgacef\\QGECAF_REVIEWER_REVISION_NESTED_BGWO', qgecaf_source='D:\\chaitali\\qgacef\\QGECAF_G2_Matched_Rerun_Package\\QGECAF_Reviewer_Revision_Nested_BGWO.py', controls_root='D:\\chaitali\\qgacef\\QGECAF_MATCHED_GATE_CONTROLS', output='D:\\chaitali\\qgacef\\QGECAF_G2_MATCHED_RERUN', image_size=128, num_classes=6, outer_folds=10, seed=42, embed_dim=64, attention_heads=4, batch_size=8, epochs=50, learning_rate=0.0001, kl_weight=0.001, dropout=0.3, fine_tune_last=30, early_stopping_patience=8, gate_hidden_units=16, fixed_gate_lambda=0.5, ece_bins=15, lower_quality_percentile=33.0, upper_quality_percentile=67.0, resume=True, smoke_test=False)

In [5]:
qroot = Path(cfg.qgecaf_root)
croot = Path(cfg.controls_root)
required = [
    Path(cfg.qgecaf_source),
    qroot / 'locked_patient_fold_assignments.csv',
    qroot / 'raw_features_presplit.npz',
    qroot / 'fold_metrics.csv',
    qroot / 'class_names.json',
    croot / 'all_gate_control_fold_metrics.csv',
    croot / 'quality_stratified_gate_controls.csv',
    croot / 'RUN_COMPLETE.json',
]
for fold in range(1, 11):
    qfold = qroot / 'cross_validation' / f'fold_{fold:02d}'
    required.extend([
        qfold / 'patient_partition_audit.csv',
        qfold / 'preprocessing.joblib',
        qfold / 'outer_holdout_predictions.csv',
    ])
    for variant in ('G0_fixed_0p5', 'G1_learned_global', 'G3_shuffled_quality'):
        required.append(croot / variant / 'cross_validation' / f'fold_{fold:02d}' / 'outer_holdout_predictions.csv')
missing = [path for path in required if not path.is_file()]
print('Required files:', len(required))
print('Missing files:', len(missing))
for path in missing[:30]:
    print('MISSING:', path)
assert not missing, 'Resolve the missing files before running G2-R.'
assert Path(cfg.output).resolve() != Path(cfg.controls_root).resolve()
assert Path(cfg.output).resolve() != Path(cfg.qgecaf_root).resolve()
print('PASS: original results are available and the new output is separate.')

Required files: 68
Missing files: 0
PASS: original results are available and the new output is separate.


## Run or safely resume

This performs ten new training runs only: one `G2_matched_rerun` for each locked outer fold. Completed G2-R folds are verified and skipped.

In [6]:
run(cfg)

TensorFlow 2.10.0; GPUs detected: 1

MATCHED GATE CONTROLS — OUTER FOLD 1/10

--- G2_matched_rerun ---
Instructions for updating:
Use `tf.compat.v1.graph_util.tensor_shape_from_node_def_name`

FINAL QGECAF MODEL COMPLEXITY
Total parameters          : 2,422,727
Trainable parameters      : 1,691,143
Non-trainable parameters  : 731,584
Weight memory             : 9.242 MiB
Network GFLOPs/image      : 0.198366
Approximate GMACs/image   : 0.099183
Epoch 1/50
1142/1143 [============================>.] - ETA: 0s - loss: 0.6931 - accuracy: 0.7870
Epoch 1: val_loss improved from inf to 0.58882, saving model to D:\chaitali\qgacef\QGECAF_G2_MATCHED_RERUN\G2_matched_rerun\cross_validation\fold_01\best_model.keras
1143/1143 [==============================] - 63s 36ms/step - loss: 0.6930 - accuracy: 0.7869 - val_loss: 0.5888 - val_accuracy: 0.7140 - lr: 1.0000e-04
Epoch 2/50
1143/1143 [==============================] - ETA: 0s - loss: 0.3640 - accuracy: 0.9013
Epoch 2: val_loss improved from 0.58882

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



1130/1130 [==============================] - ETA: 0s - loss: 0.0536 - accuracy: 0.9890
Epoch 14: val_loss did not improve from 0.19671
1130/1130 [==============================] - 38s 34ms/step - loss: 0.0536 - accuracy: 0.9890 - val_loss: 0.2757 - val_accuracy: 0.9224 - lr: 1.0000e-04
Epoch 15/50
1129/1130 [============================>.] - ETA: 0s - loss: 0.0524 - accuracy: 0.9883
Epoch 15: val_loss did not improve from 0.19671
1130/1130 [==============================] - 36s 32ms/step - loss: 0.0524 - accuracy: 0.9883 - val_loss: 0.2626 - val_accuracy: 0.9152 - lr: 1.0000e-04
Epoch 16/50
1129/1130 [============================>.] - ETA: 0s - loss: 0.0502 - accuracy: 0.9877
Epoch 16: val_loss did not improve from 0.19671

Epoch 16: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.
1130/1130 [==============================] - 36s 32ms/step - loss: 0.0502 - accuracy: 0.9877 - val_loss: 0.2314 - val_accuracy: 0.9007 - lr: 1.0000e-04
Epoch 17/50
1129/1130 [==============

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [7]:
result_root = Path(cfg.output)
display(pd.read_csv(result_root / 'gate_controls_with_G2R_mean_sd_ci95.csv'))
display(pd.read_csv(result_root / 'G2R_vs_controls_wilcoxon_holm.csv'))
display(pd.read_csv(result_root / 'G2R_vs_original_G2_verification.csv'))
display(pd.read_csv(result_root / 'pooled_gate_distribution_summary.csv'))
display(pd.read_csv(result_root / 'gate_quality_spearman_summary.csv'))
display(pd.read_csv(result_root / 'gate_lambda_quality_group_wilcoxon_holm.csv'))
print('Run complete:', (result_root / 'RUN_COMPLETE.json').is_file())

,variant,metric,folds,mean,standard_deviation,ci95_low,ci95_high
0,G0_fixed_0p5,accuracy,10,0.929242,0.012791,0.920092,0.938393
1,G0_fixed_0p5,balanced_accuracy,10,0.932634,0.012438,0.923736,0.941532
2,G0_fixed_0p5,macro_f1,10,0.915906,0.012673,0.906840,0.924971
3,G0_fixed_0p5,weighted_f1,10,0.929527,0.012781,0.920384,0.938670
4,G0_fixed_0p5,mcc,10,0.910134,0.014741,0.899589,0.920680
5,G0_fixed_0p5,ece_15_bins,10,0.028741,0.017065,0.016534,0.040949
6,G0_fixed_0p5,brier_multiclass,10,0.117339,0.018828,0.103870,0.130808
7,G0_fixed_0p5,negative_log_likelihood,10,0.293419,0.040758,0.264262,0.322575
8,G0_fixed_0p5,uncertainty_error_auroc,10,0.736227,0.064367,0.690182,0.782273
9,G1_learned_global,accuracy,10,0.929736,0.010952,0.921901,0.937571


,metric,reference,comparison,folds,mean_difference_G2R_minus_control,wilcoxon_statistic,raw_p,rank_biserial_G2R_minus_control,holm_adjusted_p_within_metric,significant_at_0.05
0,accuracy,G2_matched_rerun,G0_fixed_0p5,10,-0.002502,19.0,0.678402,-0.155556,1.000000,False
1,accuracy,G2_matched_rerun,G1_learned_global,10,-0.002996,10.0,0.262618,-0.444444,0.787855,False
2,accuracy,G2_matched_rerun,G3_shuffled_quality,10,-0.001745,24.0,0.769531,-0.127273,1.000000,False
3,balanced_accuracy,G2_matched_rerun,G0_fixed_0p5,10,0.004485,16.0,0.275391,0.418182,0.826172,False
4,balanced_accuracy,G2_matched_rerun,G1_learned_global,10,-0.001730,20.0,0.492188,-0.272727,0.984375,False
5,balanced_accuracy,G2_matched_rerun,G3_shuffled_quality,10,0.000427,27.0,1.000000,0.018182,1.000000,False
6,macro_f1,G2_matched_rerun,G0_fixed_0p5,10,-0.002230,24.0,0.769531,-0.127273,1.000000,False
7,macro_f1,G2_matched_rerun,G1_learned_global,10,-0.004492,15.0,0.232422,-0.454545,0.697266,False
8,macro_f1,G2_matched_rerun,G3_shuffled_quality,10,-0.002664,24.0,0.769531,-0.127273,1.000000,False
9,weighted_f1,G2_matched_rerun,G0_fixed_0p5,10,-0.002061,24.0,0.769531,-0.127273,1.000000,False


,metric,reference,comparison,folds,original_G2_mean,G2_matched_rerun_mean,mean_difference_G2R_minus_original_G2,wilcoxon_statistic,raw_p,rank_biserial_G2R_minus_original_G2
0,accuracy,G2_matched_rerun,G2_correct_quality,10,0.925976,0.926740,0.000765,20.0,0.767097,0.111111
1,balanced_accuracy,G2_matched_rerun,G2_correct_quality,10,0.935963,0.937120,0.001157,26.0,0.921875,0.054545
2,macro_f1,G2_matched_rerun,G2_correct_quality,10,0.911895,0.913676,0.001780,23.0,0.695312,0.163636
3,weighted_f1,G2_matched_rerun,G2_correct_quality,10,0.926756,0.927466,0.000709,25.0,0.845703,0.090909
4,mcc,G2_matched_rerun,G2_correct_quality,10,0.906647,0.907587,0.000941,23.0,0.695312,0.163636
5,ece_15_bins,G2_matched_rerun,G2_correct_quality,10,0.029862,0.033101,0.003238,25.0,0.845703,0.090909
6,brier_multiclass,G2_matched_rerun,G2_correct_quality,10,0.121129,0.120771,-0.000358,25.0,0.845703,-0.090909
7,negative_log_likelihood,G2_matched_rerun,G2_correct_quality,10,0.298073,0.297388,-0.000685,26.0,0.921875,-0.054545
8,uncertainty_error_auroc,G2_matched_rerun,G2_correct_quality,10,0.746023,0.755780,0.009757,19.0,0.431641,0.309091


,variant,quality_group,images,patients,gate_lambda_mean,gate_lambda_sample_SD,gate_lambda_median,gate_lambda_q25,gate_lambda_q75,gate_lambda_IQR,gate_lambda_minimum,gate_lambda_maximum
0,G0_fixed_0p5,Low,3732,1074,0.500000,0.000000,0.500000,0.500000,0.500000,0.000000,0.500000,0.500000
1,G0_fixed_0p5,High,3729,930,0.500000,0.000000,0.500000,0.500000,0.500000,0.000000,0.500000,0.500000
2,G0_fixed_0p5,Medium,3804,1060,0.500000,0.000000,0.500000,0.500000,0.500000,0.000000,0.500000,0.500000
3,G1_learned_global,Low,3732,1074,0.542813,0.002640,0.542866,0.541269,0.544682,0.003413,0.537471,0.547651
4,G1_learned_global,High,3729,930,0.543155,0.002739,0.543132,0.541269,0.544682,0.003413,0.537471,0.547651
5,G1_learned_global,Medium,3804,1060,0.542891,0.002593,0.542866,0.541269,0.544682,0.003413,0.537471,0.547651
6,G2_correct_quality,Low,3732,1074,0.676549,0.097776,0.683994,0.604093,0.741231,0.137138,0.144779,0.952210
7,G2_correct_quality,High,3729,930,0.657096,0.109758,0.662285,0.604557,0.722684,0.118127,0.152615,0.980302
8,G2_correct_quality,Medium,3804,1060,0.668778,0.088484,0.667368,0.612709,0.726679,0.113970,0.233651,0.949241
9,G2_matched_rerun,Low,3732,1074,0.674898,0.093804,0.674279,0.608847,0.738579,0.129732,0.152196,0.960982


,variant,relationship,folds,mean_spearman_rho,sample_SD,ci95_low,ci95_high
0,G0_fixed_0p5,actual_quality_vs_lambda,0,NaN,NaN,NaN,NaN
1,G0_fixed_0p5,gate_input_quality_vs_lambda,0,NaN,NaN,NaN,NaN
2,G1_learned_global,actual_quality_vs_lambda,0,NaN,NaN,NaN,NaN
3,G1_learned_global,gate_input_quality_vs_lambda,0,NaN,NaN,NaN,NaN
4,G2_correct_quality,actual_quality_vs_lambda,10,-0.110483,0.440831,-0.425834,0.204869
5,G2_correct_quality,gate_input_quality_vs_lambda,10,-0.110483,0.440831,-0.425834,0.204869
6,G2_matched_rerun,actual_quality_vs_lambda,10,-0.105265,0.444739,-0.423413,0.212882
7,G2_matched_rerun,gate_input_quality_vs_lambda,10,-0.105265,0.444739,-0.423413,0.212882
8,G3_shuffled_quality,actual_quality_vs_lambda,10,0.004511,0.031386,-0.017942,0.026963
9,G3_shuffled_quality,gate_input_quality_vs_lambda,10,-0.075682,0.162129,-0.191662,0.040298


,variant,comparison,folds,mean_gate_lambda_difference,wilcoxon_statistic,raw_p,rank_biserial,holm_adjusted_p_within_variant,significant_at_0.05
0,G1_learned_global,Low_minus_Medium,10,-3.330669e-17,0.0,0.179712,-1.000000,0.539137,False
1,G1_learned_global,Low_minus_High,10,-3.330669e-17,3.0,0.179712,-0.600000,0.539137,False
2,G1_learned_global,Medium_minus_High,10,0.000000e+00,5.0,1.000000,0.000000,1.000000,False
3,G2_correct_quality,Low_minus_Medium,10,1.141958e-02,19.0,0.431641,0.309091,1.000000,False
4,G2_correct_quality,Low_minus_High,10,2.236308e-02,20.0,0.492188,0.272727,1.000000,False
5,G2_correct_quality,Medium_minus_High,10,1.094350e-02,19.0,0.431641,0.309091,1.000000,False
6,G2_matched_rerun,Low_minus_Medium,10,8.067139e-03,21.0,0.556641,0.236364,1.000000,False
7,G2_matched_rerun,Low_minus_High,10,1.723424e-02,22.0,0.625000,0.200000,1.000000,False
8,G2_matched_rerun,Medium_minus_High,10,9.167105e-03,18.0,0.375000,0.345455,1.000000,False
9,G3_shuffled_quality,Low_minus_Medium,10,-5.806383e-04,21.0,0.556641,-0.236364,1.000000,False


Run complete: True
